In [1]:
!nvidia-smi

Thu Jun  4 11:07:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q ultralytics roboflow dagshub mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/6

In [3]:
from roboflow import Roboflow
import ultralytics
ultralytics.checks()

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (4 CPUs, 31.4 GB RAM, 6960.7/8062.4 GB disk)


In [4]:
import dagshub
import mlflow
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
token = user_secrets.get_secret("dagshub")
from ultralytics import settings

dagshub.auth.add_app_token(token)
dagshub.init(repo_owner='AgabaEmbedded', repo_name='Soccer-Tracking')

# Optional: Set the experiment name
mlflow.set_experiment("tunning")


settings.update({'mlflow': True})

Accessing as AgabaEmbedded

Initialized MLflow to track repo "AgabaEmbedded/Soccer-Tracking"

Repository AgabaEmbedded/Soccer-Tracking initialized!

In [5]:
from roboflow import Roboflow
rf = Roboflow(api_key="9VCfGRMmzvi61B4c9qu5")
project = rf.workspace("agabaembedded").project("soccer-tracking-large")
version = project.version(1)
dataset = version.download("yolo26")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Soccer-Tracking-Large-1 in yolo26:: 100%|██████████| 17105/17105 [00:05<00:00, 3236.02it/s] 


In [6]:
!yolo train \
model=yolo26m.pt \
name=Large-Images \
data=/kaggle/working/Soccer-Tracking-Large-1/data.yaml \
epochs=150 \
imgsz=1088,1920 \
device=0,1 \
plots=True \
batch=4 \
resume=True \
pretrained=True \
save=True \
time=5 \
deterministic=False \
optimizer=SGD \
cos_lr=True \
lr0=0.00683 \
lrf=0.00933 \
momentum=0.88752 \
weight_decay=0.00049 \
warmup_epochs=4.35648 \
warmup_momentum=0.55489 \
box=3 \
cls=0.83514 \
dfl=1.20718 \
hsv_h=0.0 \
hsv_s=0.0 \
hsv_v=0.0 \
degrees=0.0 \
translate=0.0 \
scale=0.2 \
shear=0.0 \
perspective=0.0 \
flipud=0.0 \
fliplr=0.0 \
bgr=0.0 \
mosaic=0.0 \
mixup=0.0 \
cutmix=0.0 \
copy_paste=0.0

WARNING ⚠️ model 'yolo26m.pt' is not a resumable training checkpoint (missing epoch/optimizer state). Use 'resume' only to continue incomplete training. Starting new training instead.
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=3, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.83514, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/Soccer-Tracking-Large-1/data.yaml, degrees=0.0, deterministic=False, device=0,1, dfl=1.20718, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=(1088, 1920), in

In [7]:
import os
import dagshub
import mlflow
from mlflow.tracking import MlflowClient

# 1. Initialize your DagsHub connection
dagshub.init(repo_owner='AgabaEmbedded', repo_name='Soccer-Tracking')

# 2. Set the experiment name to keep things organized
mlflow.set_experiment("Large Images")

# 3. Path to your completed weights files
local_weights_path = "/kaggle/working/runs/detect/Large-Images/weights/best.pt"
last_weights_path = "/kaggle/working/runs/detect/Large-Images/weights/last.pt"

if not os.path.exists(local_weights_path):
    raise FileNotFoundError(f"Could not find weights at {local_weights_path}.")
if not os.path.exists(last_weights_path):
    raise FileNotFoundError(f"Could not find weights at {last_weights_path}.")

# 4. Start a manual MLflow run to log and register the model
with mlflow.start_run(run_name="model_upload") as run:
    
    print("Uploading weights to DagsHub Artifacts...")
    # Log BOTH files inside a single artifact directory named "weights"
    mlflow.log_artifact(local_weights_path, artifact_path="weights")
    mlflow.log_artifact(last_weights_path, artifact_path="weights")
    
    print("Registering model in the DagsHub Model Registry...")
    # Create an MLflow client to bypass strict flavor requirements
    client = MlflowClient()
    
    # Check if the registered model container already exists; if not, create it
    try:
        client.get_registered_model("Soccer-Tracking")
    except Exception:
        print("Creating new registered model container 'Soccer-Tracking'...")
        client.create_registered_model("Soccer-Tracking")
        
    # Directly force a new version from our raw artifact directory path
    source_uri = f"runs:/{run.info.run_id}/weights"
    model_version = client.create_model_version(
        name="Soccer-Tracking",
        source=source_uri,
        run_id=run.info.run_id
    )

print(f"Successfully logged and registered Version {model_version.version} of your model!")

Initialized MLflow to track repo "AgabaEmbedded/Soccer-Tracking"

Repository AgabaEmbedded/Soccer-Tracking initialized!

2026/06/04 16:10:41 INFO mlflow.tracking.fluent: Experiment with name 'Large Images' does not exist. Creating a new experiment.


Uploading weights to DagsHub Artifacts...
Registering model in the DagsHub Model Registry...


2026/06/04 16:10:51 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Soccer-Tracking, version 3


🏃 View run model_upload at: https://dagshub.com/AgabaEmbedded/Soccer-Tracking.mlflow/#/experiments/4/runs/c7ddd814d816409c8d63961bf9308be5
🧪 View experiment at: https://dagshub.com/AgabaEmbedded/Soccer-Tracking.mlflow/#/experiments/4
Successfully logged and registered Version 3 of your model!


In [8]:
!yolo val model=/kaggle/working/runs/detect/Large-Images/weights/best.pt data=/kaggle/working/Soccer-Tracking-Large-1/data.yaml split=val imgsz=640 batch=16

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26m summary (fused): 132 layers, 20,352,536 parameters, 0 gradients, 67.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3038.5±615.4 MB/s, size: 262.0 KB)
val: Scanning /kaggle/working/Soccer-Tracking-Large-1/valid/labels.cache... 855 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 855/855 149.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 54/54 3.0it/s 18.0s
                   all        855      14671      0.679      0.554      0.599      0.316
                  ball        808        819      0.582       0.23      0.229     0.0635
            goalkeeper        480        480      0.729      0.404      0.575      0.305
                player        855      12121      0.604      0.902      0.857      0.494
               referee        800       1251      0.799       0.68      0.737        0.4
Speed: 0.6ms prepr

In [9]:
!yolo val model=/kaggle/working/runs/detect/Large-Images/weights/last.pt data=/kaggle/working/Soccer-Tracking-Large-1/data.yaml split=val imgsz=640 batch=16

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26m summary (fused): 132 layers, 20,352,536 parameters, 0 gradients, 67.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3320.8±334.2 MB/s, size: 257.2 KB)
val: Scanning /kaggle/working/Soccer-Tracking-Large-1/valid/labels.cache... 855 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 855/855 179.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 54/54 2.8it/s 19.0s
                   all        855      14671      0.679      0.554      0.599      0.316
                  ball        808        819      0.582       0.23      0.229     0.0635
            goalkeeper        480        480      0.729      0.404      0.575      0.305
                player        855      12121      0.604      0.902      0.857      0.494
               referee        800       1251      0.799       0.68      0.737        0.4
Speed: 0.6ms prepr

In [10]:
!yolo val model=/kaggle/working/runs/detect/Large-Images/weights/best.pt data=/kaggle/working/Soccer-Tracking-Large-1/data.yaml split=test imgsz=640 batch=16

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26m summary (fused): 132 layers, 20,352,536 parameters, 0 gradients, 67.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2489.2±313.6 MB/s, size: 235.3 KB)
val: Scanning /kaggle/working/Soccer-Tracking-Large-1/test/labels... 855 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 855/855 1.1Kit/s 0.8s
val: New cache created: /kaggle/working/Soccer-Tracking-Large-1/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 54/54 2.7it/s 19.9s
                   all        855      14591      0.676      0.559      0.601      0.314
                  ball        820        840      0.571      0.224      0.222     0.0624
            goalkeeper        460        460      0.735      0.422      0.585      0.309
                player        855      12061      0.603      0.903      0.862      0.492
               referee        797 

In [11]:
!yolo val model=/kaggle/working/runs/detect/Large-Images/weights/last.pt data=/kaggle/working/Soccer-Tracking-Large-1/data.yaml split=test imgsz=640 batch=16

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26m summary (fused): 132 layers, 20,352,536 parameters, 0 gradients, 67.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2340.6±454.3 MB/s, size: 247.0 KB)
val: Scanning /kaggle/working/Soccer-Tracking-Large-1/test/labels.cache... 855 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 855/855 155.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 54/54 2.8it/s 19.2s
                   all        855      14591      0.676      0.559      0.601      0.314
                  ball        820        840      0.571      0.224      0.222     0.0624
            goalkeeper        460        460      0.735      0.422      0.585      0.309
                player        855      12061      0.603      0.903      0.862      0.492
               referee        797       1230      0.795      0.687      0.736      0.395
Speed: 0.6ms prepro

In [12]:
!yolo export model=/kaggle/working/yolo11m.pt format=torchscript

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11m summary (fused): 125 layers, 20,091,712 parameters, 0 gradients, 68.0 GFLOPs

PyTorch: starting from '/kaggle/working/yolo11m.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (38.8 MB)

TorchScript: starting export with torch 2.10.0+cu128...
TorchScript: export success ✅ 5.4s, saved as '/kaggle/working/yolo11m.torchscript' (77.3 MB)

Export complete (7.2s)
Results saved to /kaggle/working/yolo11m.torchscript
Predict:         yolo predict task=detect model=/kaggle/working/yolo11m.torchscript imgsz=640 
Validate:        yolo val task=detect model=/kaggle/working/yolo11m.torchscript imgsz=640 data=/ultralytics/ultralytics/cfg/datasets/coco.yaml  
Visualize:       https://netron.app
💡 Learn more at https://docs.ultralytics.com/mod